In [ ]:
import asyncio
import io
import aiohttp
from PIL import Image

image_urls = [
    # Credit: Scott Foster
    "https://neont.s3.amazonaws.com/wp-content/uploads/2017/11/Scott-Foster-Too-Pooped-to-Party.jpg",
    # Credit: Katie McPherson
    "https://storageciggallery.addons.business/13611/cig-cozy-gallery-6892vUf-Katie-McPherson-CA-hd.jpg?c=00",
    # Credit: Kyle Finger
    "https://storageciggallery.addons.business/13611/cig-cozy-gallery-6892nr8-KyleFingerWisconsin-hd.jpg?c=00",
    # Credit: Cayuga Nature Center
    "https://images.squarespace-cdn.com/content/v1/5d3cb13b96f9ac0001e89cf6/1578338468978-4U1LVRPRLU2B83UR77BX/Smith-Woods-trees.JPG",
]


async def fetch_image(session: aiohttp.ClientSession, url: str) -> Image.Image:
    async with session.get(url) as response:
        image_bytes = await response.read()
        return Image.open(io.BytesIO(image_bytes))


async def fetch_images(urls: list[str]) -> list[Image.Image]:
    async with aiohttp.ClientSession() as session:
        return await asyncio.gather(*[fetch_image(session, url) for url in urls])


def get_bytes_from_image(image: Image.Image) -> bytes:
    img_byte_arr = io.BytesIO()
    image.save(img_byte_arr, format="JPEG")
    return img_byte_arr.getvalue()


images = await fetch_images(image_urls)

In [ ]:
from IPython.display import display  # pyright: ignore

for image in images:
    display(image)

In [ ]:
from dragoneye import Dragoneye, Image as DragoneyeImage, PredictionTaskError

MODEL_NAME = "recognize_anything/animals"
# Fill this in with your Dragoneye API key (or set the DRAGONEYE_API_KEY env var
# and use Dragoneye() with no arguments).
API_KEY = "<YOUR_API_KEY>"

SETUP_URL = (
    "https://github.com/dragoneyeAI/dragoneye-cookbooks/tree/main/"
    "trail-camera#before-you-start-create-your-animals-model"
)

client = Dragoneye(api_key=API_KEY)


async def get_prediction(image: Image.Image):
    try:
        return await client.classification.predict_image(
            media=DragoneyeImage.from_bytes(
                get_bytes_from_image(image), mime_type="image/jpeg"
            ),
            model_name=MODEL_NAME,
        )
    except PredictionTaskError as error:
        # A 404 from the API means the model name doesn't exist on your account yet.
        underlying_status = (
            getattr(error.args[1], "status", None) if len(error.args) > 1 else None
        )
        if underlying_status == 404 or "404" in str(error):
            raise RuntimeError(
                f"Model '{MODEL_NAME}' was not found. You need to create it before "
                f"running this cookbook. Follow the setup instructions in the README: "
                f"{SETUP_URL}"
            ) from None
        raise


prediction_results = await asyncio.gather(*[get_prediction(image) for image in images])
images_with_prediction_results = list(zip(images, prediction_results))

In [ ]:
from dragoneye import ClassificationPredictImageResponse


def all_objects_in_image(
    prediction_result: ClassificationPredictImageResponse,
    prediction_threshold: float = 0.7,
) -> list[tuple[str, float]]:
    results: list[tuple[str, float]] = []
    for detected_object in prediction_result.objects:
        if not detected_object.categories:
            continue
        # Each detected object's best-scoring category is the animal it recognized.
        top_category = max(
            detected_object.categories, key=lambda category: category.score
        )
        if top_category.score > prediction_threshold:
            results.append((top_category.name, top_category.score))
    return results


images_with_objects_in_image = [
    (image, all_objects_in_image(prediction_result))
    for image, prediction_result in images_with_prediction_results
]

In [ ]:
for image, objects_in_image in images_with_objects_in_image:
    print(f"Animals detected in image: {set([name for name, _ in objects_in_image])}")
    display(image)